In [1]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q sentence-transformers faiss-cpu rouge-score tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 130.9 MB/s eta 0:00:00


In [3]:
"""
=============================================================================
ROUGE-ALIGNED FINE-TUNING of AfriE5
=============================================================================
WHY this works:
- Current AfriE5: ranks by question similarity
- Problem: similar question ≠ similar answer wording
- Fix: teach AfriE5 "this question is the right match because its ANSWER
  has the highest ROUGE overlap with the reference"
- This directly aligns retrieval with the evaluation metric
=============================================================================
"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['USE_TF'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import torch
import faiss
import gc
import re
from pathlib import Path
from tqdm import tqdm
from rouge_score import rouge_scorer
from datetime import datetime
from collections import defaultdict
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)
# ============================================================
# PATHS
# ============================================================
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        log("Google Drive already mounted.")
    DATA_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/data')
    OUTPUT_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/outputs')
    AFRIE5_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/outputs/afrie5-final-model')
except ImportError:
    DATA_DIR = Path('data/raw/')
    OUTPUT_DIR = Path('outputs/')
    AFRIE5_DIR = None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================
# LOAD DATA
# ============================================================
log("Loading data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
val_df   = pd.read_csv(DATA_DIR / 'Val.csv')
test_df  = pd.read_csv(DATA_DIR / 'Test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    log(f"  {name}: {len(df)} rows")
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input', 'output'])
combined = combined.reset_index(drop=True)
questions_raw = combined['input'].fillna('').astype(str).tolist()
answers_raw   = combined['output'].fillna('').astype(str).tolist()
log(f"Combined corpus: {len(combined)} samples")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
if torch.cuda.is_available():
    log(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# ============================================================
# FAST ROUGE (100x faster than rouge_score for data building)
# ============================================================
def fast_rouge1_f1(ref_str, hyp_str):
    """Fast ROUGE-1 F1 using set overlap. ~100x faster than rouge_score."""
    ref_tokens = set(ref_str.lower().split())
    hyp_tokens = set(hyp_str.lower().split())
    if not ref_tokens or not hyp_tokens:
        return 0.0
    overlap = len(ref_tokens & hyp_tokens)
    if overlap == 0:
        return 0.0
    precision = overlap / len(hyp_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)
# ============================================================
# STEP 1: LOAD MODEL + BUILD INDEX
# ============================================================
log(f"\n{'='*60}")
log("STEP 1: Load AfriE5 + Build index")
log(f"{'='*60}")
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
PREFIX = "query: "
if AFRIE5_DIR and AFRIE5_DIR.exists():
    bienc = SentenceTransformer(str(AFRIE5_DIR), device='cuda:0')
    log(f"AfriE5 loaded from Drive: {sum(p.numel() for p in bienc.parameters())/1e6:.0f}M params")
else:
    bienc = SentenceTransformer('McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
    log(f"AfriE5 from HuggingFace: {sum(p.numel() for p in bienc.parameters())/1e6:.0f}M params")
log("Encoding corpus questions...")
corpus_emb = bienc.encode(
    [f"{PREFIX}{q}" for q in questions_raw],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
fidx = faiss.IndexFlatIP(corpus_emb.shape[1])
fidx.add(corpus_emb)
log(f"Index: {corpus_emb.shape}")
# ============================================================
# STEP 2: MEASURE CURRENT BASELINE ON VAL
# ============================================================
log(f"\n{'='*60}")
log("STEP 2: Current baseline on val")
log(f"{'='*60}")
val_qs = val_df['input'].fillna('').astype(str).tolist()
val_emb = bienc.encode(
    [f"{PREFIX}{q}" for q in val_qs],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
baseline_r1s, baseline_rls = [], []
per_lang_base = defaultdict(lambda: {'r1': [], 'rl': []})
for i in tqdm(range(len(val_df)), desc="Baseline eval"):
    q = str(val_df.iloc[i]['input']).strip()
    ref = str(val_df.iloc[i]['output']).strip()
    sub = str(val_df.iloc[i]['subset'])
    if not ref: continue
    D, I = fidx.search(val_emb[i:i+1], 5)
    answer = ''
    for j in range(5):
        ci = int(I[0][j])
        if ci >= len(combined): continue
        if str(combined.iloc[ci]['input']).strip() == q: continue
        answer = str(combined.iloc[ci]['output'])
        break
    if not answer:
        answer = str(combined.iloc[int(I[0][0])]['output'])
    r = scorer.score(ref, answer)
    baseline_r1s.append(r['rouge1'].fmeasure)
    baseline_rls.append(r['rougeL'].fmeasure)
    per_lang_base[sub]['r1'].append(r['rouge1'].fmeasure)
    per_lang_base[sub]['rl'].append(r['rougeL'].fmeasure)
b_r1, b_rl = np.mean(baseline_r1s), np.mean(baseline_rls)
log(f"\nBaseline: R1={b_r1:.4f} RL={b_rl:.4f}")
for sub in sorted(per_lang_base.keys()):
    d = per_lang_base[sub]
    log(f"  {sub:<12} R1={np.mean(d['r1']):.4f} RL={np.mean(d['rl']):.4f}")
# ============================================================
# STEP 3: BUILD ROUGE-ALIGNED TRAINING DATA
# ============================================================
log(f"\n{'='*60}")
log("STEP 3: Build ROUGE-aligned training data")
log(f"{'='*60}")
log("For each train Q, finding which top-30 candidate has the")
log("BEST ROUGE overlap → that becomes the positive pair.")
train_qs_text = train_df['input'].fillna('').astype(str).tolist()
train_as_text = train_df['output'].fillna('').astype(str).tolist()
log("Encoding train questions...")
train_q_emb = bienc.encode(
    [f"{PREFIX}{q}" for q in train_qs_text],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
CAND_K = 30  # candidates per query
rouge_aligned_pairs = []
improvements = 0
total_checked = 0
for i in tqdm(range(len(train_df)), desc="ROUGE-aligned pairs"):
    q = train_qs_text[i].strip()
    ref = train_as_text[i].strip()
    if not q or not ref:
        continue
    D, I = fidx.search(train_q_emb[i:i+1], CAND_K + 5)
    # Get top-1 (current best) and all candidates
    candidates = []
    for j in range(CAND_K + 5):
        ci = int(I[0][j])
        if ci >= len(combined): continue
        cq = str(combined.iloc[ci]['input']).strip()
        if cq == q: continue  # skip self
        ca = str(combined.iloc[ci]['output']).strip()
        candidates.append((ci, cq, ca))
        if len(candidates) >= CAND_K: break
    if len(candidates) < 3:
        continue
    # Compute fast ROUGE for each candidate's answer vs reference
    rouge_scores = []
    for ci, cq, ca in candidates:
        r1 = fast_rouge1_f1(ref, ca)
        rouge_scores.append(r1)
    # Current top-1 ROUGE
    top1_rouge = rouge_scores[0]
    # Best ROUGE candidate
    best_idx = int(np.argmax(rouge_scores))
    best_rouge = rouge_scores[best_idx]
    best_ci, best_cq, best_ca = candidates[best_idx]
    total_checked += 1
    # Only create training pair if the best candidate is NOT already top-1
    # AND the improvement is meaningful (> 0.05 ROUGE)
    if best_idx > 0 and best_rouge - top1_rouge > 0.05:
        # Positive pair: query → best ROUGE question
        rouge_aligned_pairs.append(InputExample(
            texts=[f"{PREFIX}{q}", f"{PREFIX}{best_cq}"]
        ))
        improvements += 1
    # Also add the top-1 as positive (to maintain what's already correct)
    top1_ci, top1_cq, top1_ca = candidates[0]
    if top1_rouge > 0.3:  # only if top-1 is decent
        rouge_aligned_pairs.append(InputExample(
            texts=[f"{PREFIX}{q}", f"{PREFIX}{top1_cq}"]
        ))
log(f"\nROUGE-aligned training examples: {len(rouge_aligned_pairs)}")
log(f"Cases where current top-1 is NOT the best: {improvements}/{total_checked} "
    f"({100*improvements/max(total_checked,1):.1f}%)")
# Free embeddings
del train_q_emb
gc.collect()
# ============================================================
# STEP 4: FINE-TUNE AfriE5 WITH ROUGE-ALIGNED PAIRS
# ============================================================
log(f"\n{'='*60}")
log("STEP 4: Fine-tune AfriE5 with ROUGE-aligned objective")
log(f"{'='*60}")
train_loader = DataLoader(rouge_aligned_pairs, batch_size=16, shuffle=True)
train_loss = losses.MultipleNegativesRankingLoss(bienc)
# Warmup steps
total_steps = len(train_loader) * 1  # 1 epoch
warmup_steps = int(total_steps * 0.1)
log(f"Training examples: {len(rouge_aligned_pairs)}")
log(f"Batch size: 16")
log(f"Total steps: {total_steps}")
log(f"Warmup: {warmup_steps}")
log(f"Learning rate: 1e-5 (low — preserving existing knowledge)")
bienc.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=1,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': 1e-5},
    show_progress_bar=True,
    output_path=str(OUTPUT_DIR / 'afrie5-rouge-aligned'),
)
log("Training complete!")
bienc.save(str(OUTPUT_DIR / 'afrie5-rouge-aligned'))
log("Model saved to Drive!")
# ============================================================
# STEP 5: EVALUATE IMPROVED MODEL ON VAL
# ============================================================
log(f"\n{'='*60}")
log("STEP 5: Evaluate ROUGE-aligned model on val")
log(f"{'='*60}")
# Re-encode everything with the improved model
log("Re-encoding corpus...")
corpus_emb_new = bienc.encode(
    [f"{PREFIX}{q}" for q in questions_raw],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
fidx_new = faiss.IndexFlatIP(corpus_emb_new.shape[1])
fidx_new.add(corpus_emb_new)
log("Re-encoding val...")
val_emb_new = bienc.encode(
    [f"{PREFIX}{q}" for q in val_qs],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
new_r1s, new_rls = [], []
per_lang_new = defaultdict(lambda: {'r1': [], 'rl': []})
for i in tqdm(range(len(val_df)), desc="New model eval"):
    q = str(val_df.iloc[i]['input']).strip()
    ref = str(val_df.iloc[i]['output']).strip()
    sub = str(val_df.iloc[i]['subset'])
    if not ref: continue
    D, I = fidx_new.search(val_emb_new[i:i+1], 5)
    answer = ''
    for j in range(5):
        ci = int(I[0][j])
        if ci >= len(combined): continue
        if str(combined.iloc[ci]['input']).strip() == q: continue
        answer = str(combined.iloc[ci]['output'])
        break
    if not answer:
        answer = str(combined.iloc[int(I[0][0])]['output'])
    r = scorer.score(ref, answer)
    new_r1s.append(r['rouge1'].fmeasure)
    new_rls.append(r['rougeL'].fmeasure)
    per_lang_new[sub]['r1'].append(r['rouge1'].fmeasure)
    per_lang_new[sub]['rl'].append(r['rougeL'].fmeasure)
n_r1, n_rl = np.mean(new_r1s), np.mean(new_rls)
log(f"\n{'='*60}")
log(f"{'Method':<30} {'ROUGE-1':>10} {'ROUGE-L':>10}")
log(f"{'-'*52}")
log(f"{'Before (baseline)':30} {b_r1:>10.4f} {b_rl:>10.4f}")
log(f"{'After (ROUGE-aligned)':30} {n_r1:>10.4f} {n_rl:>10.4f}")
log(f"{'IMPROVEMENT':30} {n_r1-b_r1:>+10.4f} {n_rl-b_rl:>+10.4f}")
log(f"{'='*60}")
log(f"\nPer-language comparison:")
log(f"{'Subset':<12} {'Base R1':>8} {'New R1':>8} {'Δ':>7} | {'Base RL':>8} {'New RL':>8} {'Δ':>7}")
for sub in sorted(set(list(per_lang_base.keys()) + list(per_lang_new.keys()))):
    br1 = np.mean(per_lang_base[sub]['r1']) if per_lang_base[sub]['r1'] else 0
    brl = np.mean(per_lang_base[sub]['rl']) if per_lang_base[sub]['rl'] else 0
    nr1 = np.mean(per_lang_new[sub]['r1']) if per_lang_new[sub]['r1'] else 0
    nrl = np.mean(per_lang_new[sub]['rl']) if per_lang_new[sub]['rl'] else 0
    log(f"  {sub:<12} {br1:>8.4f} {nr1:>8.4f} {nr1-br1:>+7.4f} | {brl:>8.4f} {nrl:>8.4f} {nrl-brl:>+7.4f}")
# ============================================================
# STEP 6: GENERATE TEST SUBMISSION (only if improved)
# ============================================================
log(f"\n{'='*60}")
log("STEP 6: Generate test submission")
log(f"{'='*60}")
if n_r1 + n_rl > b_r1 + b_rl:
    log("ROUGE-aligned model is BETTER! Generating submission...")
    use_index = fidx_new
    use_model_name = "rouge_aligned"
    log("Encoding test...")
    test_emb = bienc.encode(
        [f"{PREFIX}{q}" for q in test_df['input'].fillna('').astype(str).tolist()],
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    ).astype(np.float32)
else:
    log("ROUGE-aligned model is NOT better. Using baseline for submission.")
    use_index = faiss.IndexFlatIP(corpus_emb.shape[1])
    use_index.add(corpus_emb)
    use_model_name = "baseline"
    test_emb = bienc.encode(
        [f"{PREFIX}{q}" for q in test_df['input'].fillna('').astype(str).tolist()],
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    ).astype(np.float32)
test_qs = test_df['input'].fillna('').astype(str).tolist()
rows = []
for i in tqdm(range(len(test_df)), desc="Test submission"):
    q = test_qs[i].strip()
    D, I = use_index.search(test_emb[i:i+1], 5)
    answer = "No answer found."
    for j in range(5):
        ci = int(I[0][j])
        if ci >= len(combined): continue
        if str(combined.iloc[ci]['input']).strip() == q: continue
        answer = str(combined.iloc[ci]['output'])
        break
    rows.append({
        'ID': test_df.iloc[i]['ID'],
        'TargetRLF1': answer, 'TargetR1F1': answer, 'TargetLLM': answer,
    })
sub = pd.DataFrame(rows)[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
assert len(sub) == len(sample_sub)
fname = f'submission_{use_model_name}.csv'
sub.to_csv(OUTPUT_DIR / fname, index=False)
log(f"Saved: {fname}")
# ============================================================
# STEP 7: TRY SECOND EPOCH (if first helped)
# ============================================================
if n_r1 + n_rl > b_r1 + b_rl:
    log(f"\n{'='*60}")
    log("STEP 7: Training epoch 2 (pushing further)")
    log(f"{'='*60}")
    epoch1_r1, epoch1_rl = n_r1, n_rl
    # Rebuild ROUGE-aligned data with the improved model
    log("Rebuilding ROUGE-aligned data with improved embeddings...")
    train_q_emb2 = bienc.encode(
        [f"{PREFIX}{q}" for q in train_qs_text],
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    ).astype(np.float32)
    rouge_pairs_2 = []
    for i in tqdm(range(len(train_df)), desc="Epoch 2 pairs"):
        q = train_qs_text[i].strip()
        ref = train_as_text[i].strip()
        if not q or not ref: continue
        D, I = fidx_new.search(train_q_emb2[i:i+1], CAND_K + 5)
        candidates = []
        for j in range(CAND_K + 5):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            cq = str(combined.iloc[ci]['input']).strip()
            if cq == q: continue
            ca = str(combined.iloc[ci]['output']).strip()
            candidates.append((ci, cq, ca))
            if len(candidates) >= CAND_K: break
        if len(candidates) < 3: continue
        rouge_scores = [fast_rouge1_f1(ref, ca) for _, _, ca in candidates]
        best_idx = int(np.argmax(rouge_scores))
        if best_idx > 0 and rouge_scores[best_idx] - rouge_scores[0] > 0.05:
            rouge_pairs_2.append(InputExample(
                texts=[f"{PREFIX}{q}", f"{PREFIX}{candidates[best_idx][1]}"]
            ))
        if rouge_scores[0] > 0.3:
            rouge_pairs_2.append(InputExample(
                texts=[f"{PREFIX}{q}", f"{PREFIX}{candidates[0][1]}"]
            ))
    del train_q_emb2; gc.collect()
    log(f"Epoch 2 pairs: {len(rouge_pairs_2)}")
    loader_2 = DataLoader(rouge_pairs_2, batch_size=16, shuffle=True)
    bienc.fit(
        train_objectives=[(loader_2, train_loss)],
        epochs=1,
        warmup_steps=int(len(loader_2) * 0.1),
        optimizer_params={'lr': 5e-6},  # even lower LR
        show_progress_bar=True,
        output_path=str(OUTPUT_DIR / 'afrie5-rouge-aligned-v2'),
    )
    bienc.save(str(OUTPUT_DIR / 'afrie5-rouge-aligned-v2'))
    log("Epoch 2 complete! Model saved.")
    # Evaluate epoch 2
    log("Evaluating epoch 2...")
    corpus_emb_v2 = bienc.encode(
        [f"{PREFIX}{q}" for q in questions_raw],
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    ).astype(np.float32)
    fidx_v2 = faiss.IndexFlatIP(corpus_emb_v2.shape[1])
    fidx_v2.add(corpus_emb_v2)
    val_emb_v2 = bienc.encode(
        [f"{PREFIX}{q}" for q in val_qs],
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    ).astype(np.float32)
    v2_r1s, v2_rls = [], []
    for i in tqdm(range(len(val_df)), desc="Epoch 2 eval"):
        q = str(val_df.iloc[i]['input']).strip()
        ref = str(val_df.iloc[i]['output']).strip()
        if not ref: continue
        D, I = fidx_v2.search(val_emb_v2[i:i+1], 5)
        answer = ''
        for j in range(5):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            if str(combined.iloc[ci]['input']).strip() == q: continue
            answer = str(combined.iloc[ci]['output'])
            break
        if not answer: answer = str(combined.iloc[int(I[0][0])]['output'])
        r = scorer.score(ref, answer)
        v2_r1s.append(r['rouge1'].fmeasure)
        v2_rls.append(r['rougeL'].fmeasure)
    v2_r1, v2_rl = np.mean(v2_r1s), np.mean(v2_rls)
    log(f"\n{'Method':<30} {'ROUGE-1':>10} {'ROUGE-L':>10}")
    log(f"{'-'*52}")
    log(f"{'Baseline':30} {b_r1:>10.4f} {b_rl:>10.4f}")
    log(f"{'Epoch 1':30} {epoch1_r1:>10.4f} {epoch1_rl:>10.4f}")
    log(f"{'Epoch 2':30} {v2_r1:>10.4f} {v2_rl:>10.4f}")
    # Generate epoch 2 submission if better
    if v2_r1 + v2_rl > epoch1_r1 + epoch1_rl:
        log("\nEpoch 2 is better! Generating submission...")
        test_emb_v2 = bienc.encode(
            [f"{PREFIX}{q}" for q in test_df['input'].fillna('').astype(str).tolist()],
            batch_size=64, show_progress_bar=True, normalize_embeddings=True
        ).astype(np.float32)
        rows_v2 = []
        for i in tqdm(range(len(test_df)), desc="Epoch 2 submission"):
            q = test_qs[i].strip()
            D, I = fidx_v2.search(test_emb_v2[i:i+1], 5)
            answer = "No answer found."
            for j in range(5):
                ci = int(I[0][j])
                if ci >= len(combined): continue
                if str(combined.iloc[ci]['input']).strip() == q: continue
                answer = str(combined.iloc[ci]['output'])
                break
            rows_v2.append({
                'ID': test_df.iloc[i]['ID'],
                'TargetRLF1': answer, 'TargetR1F1': answer, 'TargetLLM': answer,
            })
        sub_v2 = pd.DataFrame(rows_v2)[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
        assert len(sub_v2) == len(sample_sub)
        sub_v2.to_csv(OUTPUT_DIR / 'submission_rouge_aligned_v2.csv', index=False)
        log("Saved: submission_rouge_aligned_v2.csv")
    else:
        log("Epoch 2 did NOT improve. Stick with epoch 1 submission.")
else:
    log("\nSkipping epoch 2 (epoch 1 didn't help).")
# ============================================================
# FINAL SUMMARY
# ============================================================
log(f"\n{'='*60}")
log("DONE")
log(f"{'='*60}")
log(f"Previous LB best: 0.6545")
log(f"Submissions saved to Drive:")
for f in sorted(OUTPUT_DIR.glob("submission_*.csv")):
    log(f"  → {f.name}")


[16:25:30] Google Drive already mounted.
[16:25:30] Loading data...
[16:25:34]   Train: 29815 rows
[16:25:34]   Val: 6686 rows
[16:25:34]   Test: 2618 rows
[16:25:34] Combined corpus: 36501 samples
[16:25:34] GPU: NVIDIA A100-SXM4-40GB | 42.4 GB
[16:25:34] 
[16:25:34] STEP 1: Load AfriE5 + Build index
[16:25:34] ============================================================


/tmp/ipykernel_554/1361521229.py:87: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[16:26:45] AfriE5 loaded from Drive: 560M params
[16:26:45] Encoding corpus questions...


Batches:   0%|          | 0/571 [00:00<?, ?it/s]

...

New model eval: 100%|██████████| 6686/6686 [01:10<00:00, 95.19it/s]

[16:51:53] 
[16:51:53] Method                            ROUGE-1    ROUGE-L
[16:51:53] ----------------------------------------------------
[16:51:53] Before (baseline)                  0.6293     0.5727
[16:51:53] After (ROUGE-aligned)              0.6264     0.5689
[16:51:53] IMPROVEMENT                       -0.0029    -0.0038
[16:51:53] ============================================================
[16:51:53] 
Per-language comparison:
[16:51:53] Subset        Base R1   New R1       Δ |  Base RL   New RL       Δ
[16:51:53]   Aka_Gha        0.3905   0.3952 +0.0047 |   0.2250   0.2266 +0.0016
[16:51:53]   Amh_Eth        0.0421   0.0389 -0.0032 |   0.0421   0.0389 -0.0032
[16:51:53]   Eng_Eth        0.6893   0.6843 -0.0050 |   0.6669   0.6617 -0.0052
[16:51:53]   Eng_Gha        0.3356   0.3404 +0.0049 |   0.2139   0.2184 +0.0046
[16:51:53]   Eng_Ken        0.8975   0.8948 -0.0026 |   0.8860   0.8829 -0.0030
[16:51:53]   Eng_Uga        0.8621   0.8600 -0.0021 |   0.8472   0.8450 -0.0022
[

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Test submission: 100%|██████████| 2618/2618 [00:18<00:00, 141.04it/s]


[16:52:19] Saved: submission_baseline.csv
[16:52:19] 
Skipping epoch 2 (epoch 1 didn't help).
[16:52:19] 
[16:52:19] DONE
[16:52:19] ============================================================
[16:52:19] Previous LB best: 0.6545
[16:52:19] Submissions saved to Drive:
[16:52:19]   → submission_baseline.csv
[16:52:19]   → submission_baseline_qq.csv
[16:52:19]   → submission_ce_reranked.csv
[16:52:19]   → submission_fusion_best.csv
[16:52:19]   → submission_fusion_percol.csv
[16:52:19]   → submission_rag_reader.csv
[16:52:19]   → submission_rag_reader_PARTIAL.csv


In [4]:
import pandas as pd

base = pd.read_csv('/content/drive/MyDrive/multilingual-health-qa/outputs/submission_baseline_qq.csv')
rag  = pd.read_csv('/content/drive/MyDrive/multilingual-health-qa/outputs/submission_rag_reader.csv')

# ROUGE columns from retrieval, LLM column from generated
hybrid = base.copy()
hybrid['TargetLLM'] = rag['TargetLLM']

hybrid.to_csv('/content/drive/MyDrive/multilingual-health-qa/outputs/submission_hybrid.csv', index=False)
print(f"Saved! {len(hybrid)} rows")
print(hybrid.head())

Saved! 2618 rows
                       ID                                         TargetRLF1  \
0  ID_TS_Aka_Gha_A3B1799D  yɛ nhyehyɛe ma ɔsatu ahorow a wɔde ko tia bɔbe...   
1  ID_TS_Aka_Gha_1C80317F  Hokwan a mmabun wɔ sɛ wɔma wɔn nipadua mu ahof...   
2  ID_TS_Aka_Gha_06671AD1  Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basab...   
3  ID_TS_Aka_Gha_BDD640FB  Nya nsakrae tumi 'influence'. Ɛhyehyɛ  adwen. ...   
4  ID_TS_Aka_Gha_46685257  Ɛboa ma wɔnya nna mu nsɛm ho 'SRH' mmoa (contr...   

                                          TargetR1F1  \
0  yɛ nhyehyɛe ma ɔsatu ahorow a wɔde ko tia bɔbe...   
1  Hokwan a mmabun wɔ sɛ wɔma wɔn nipadua mu ahof...   
2  Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basab...   
3  Nya nsakrae tumi 'influence'. Ɛhyehyɛ  adwen. ...   
4  Ɛboa ma wɔnya nna mu nsɛm ho 'SRH' mmoa (contr...   

                                           TargetLLM  
0                                  no ho nsɛm ka ho.  
1  ɔware, mmea awotwaa a wotwitwa, ne awo a wɔhyɛ...  


In [5]:
!pip install -q sentence-transformers faiss-cpu rouge-score tqdm

In [6]:
"""
=============================================================================
DATA-DRIVEN ANALYSIS + PER-LANGUAGE STRATEGY + GEMINI LLM COLUMN
=============================================================================
1. Analyze WHY weak languages score low (Amh 0.04, Aka 0.39, Eng_Gha 0.34)
2. Test per-language retrieval strategies (global vs same-lang vs hybrid)
3. Pick BEST strategy per language based on val data
4. Optionally: use Gemini API for LLM-Judge column
5. Generate optimized submission
=============================================================================
"""
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['USE_TF'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import torch
import faiss
import gc
import json
from pathlib import Path
from tqdm import tqdm
from rouge_score import rouge_scorer
from datetime import datetime
from collections import defaultdict, Counter
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)
# ============================================================
# PATHS + DATA
# ============================================================
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        log("Google Drive already mounted.")
    DATA_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/data')
    OUTPUT_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/outputs')
    AFRIE5_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/outputs/afrie5-final-model')
except ImportError:
    DATA_DIR = Path('data/raw/')
    OUTPUT_DIR = Path('outputs/')
    AFRIE5_DIR = None
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log("Loading data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
val_df   = pd.read_csv(DATA_DIR / 'Val.csv')
test_df  = pd.read_csv(DATA_DIR / 'Test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input', 'output'])
combined = combined.reset_index(drop=True)
questions_raw = combined['input'].fillna('').astype(str).tolist()
answers_raw   = combined['output'].fillna('').astype(str).tolist()
subsets_raw   = combined['subset'].fillna('').astype(str).tolist()
log(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Combined: {len(combined)}")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
# ============================================================
# BUILD PER-LANGUAGE INDICES
# ============================================================
log(f"\n{'='*60}")
log("STEP 1: Load model + Build global and per-language indices")
log(f"{'='*60}")
from sentence_transformers import SentenceTransformer
PREFIX = "query: "
if AFRIE5_DIR and AFRIE5_DIR.exists():
    bienc = SentenceTransformer(str(AFRIE5_DIR), device='cuda:0')
    log(f"AfriE5 from Drive: {sum(p.numel() for p in bienc.parameters())/1e6:.0f}M params")
else:
    bienc = SentenceTransformer('McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
    log(f"AfriE5 from HuggingFace")
log("Encoding corpus...")
corpus_emb = bienc.encode(
    [f"{PREFIX}{q}" for q in questions_raw],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
# Global index
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1])
global_idx.add(corpus_emb)
log(f"Global index: {corpus_emb.shape}")
# Per-language indices
lang_indices = {}   # subset -> (faiss_index, global_row_indices)
unique_subsets = sorted(set(subsets_raw))
for sub in unique_subsets:
    mask = [i for i, s in enumerate(subsets_raw) if s == sub]
    if not mask:
        continue
    sub_emb = corpus_emb[mask]
    idx = faiss.IndexFlatIP(sub_emb.shape[1])
    idx.add(sub_emb)
    lang_indices[sub] = (idx, mask)
    log(f"  {sub}: {len(mask)} samples")
# Encode val
log("Encoding val queries...")
val_qs = val_df['input'].fillna('').astype(str).tolist()
val_emb = bienc.encode(
    [f"{PREFIX}{q}" for q in val_qs],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
bienc.cpu(); gc.collect(); torch.cuda.empty_cache()
log("GPU freed.")
# ============================================================
# STEP 2: DEEP ANALYSIS OF EACH LANGUAGE
# ============================================================
log(f"\n{'='*60}")
log("STEP 2: Deep analysis of each language")
log(f"{'='*60}")
# For each val question, get top-50 global candidates and compute oracle
analysis = defaultdict(lambda: {
    'count': 0, 'r1_top1': [], 'rl_top1': [], 'r1_oracle': [], 'rl_oracle': [],
    'oracle_from_same_lang': 0, 'oracle_from_diff_lang': 0,
    'oracle_rank': [], 'avg_q_len': [], 'avg_a_len': [], 'avg_ref_len': [],
    'top1_same_lang': 0, 'top1_diff_lang': 0,
    'examples': [],
})
TOP_K = 50
for i in tqdm(range(len(val_df)), desc="Analyzing val"):
    q = str(val_df.iloc[i]['input']).strip()
    ref = str(val_df.iloc[i]['output']).strip()
    sub = str(val_df.iloc[i]['subset'])
    if not ref or not q:
        continue
    D, I = global_idx.search(val_emb[i:i+1], TOP_K + 5)
    candidates = []
    for j in range(TOP_K + 5):
        ci = int(I[0][j])
        if ci >= len(combined): continue
        if str(combined.iloc[ci]['input']).strip() == q: continue
        candidates.append(ci)
        if len(candidates) >= TOP_K: break
    if not candidates:
        continue
    # Top-1
    top1_ci = candidates[0]
    top1_answer = str(combined.iloc[top1_ci]['output'])
    top1_sub = str(combined.iloc[top1_ci]['subset'])
    r_top1 = scorer.score(ref, top1_answer)
    # Oracle (best ROUGE in top-50)
    best_r1, best_rl, best_ci, best_rank = 0, 0, candidates[0], 0
    for rank, ci in enumerate(candidates):
        ca = str(combined.iloc[ci]['output'])
        r = scorer.score(ref, ca)
        combined_r = r['rouge1'].fmeasure + r['rougeL'].fmeasure
        if combined_r > best_r1 + best_rl:
            best_r1, best_rl = r['rouge1'].fmeasure, r['rougeL'].fmeasure
            best_ci, best_rank = ci, rank
    a = analysis[sub]
    a['count'] += 1
    a['r1_top1'].append(r_top1['rouge1'].fmeasure)
    a['rl_top1'].append(r_top1['rougeL'].fmeasure)
    a['r1_oracle'].append(best_r1)
    a['rl_oracle'].append(best_rl)
    a['oracle_rank'].append(best_rank)
    a['avg_q_len'].append(len(q.split()))
    a['avg_a_len'].append(len(top1_answer.split()))
    a['avg_ref_len'].append(len(ref.split()))
    if str(combined.iloc[best_ci]['subset']) == sub:
        a['oracle_from_same_lang'] += 1
    else:
        a['oracle_from_diff_lang'] += 1
    if top1_sub == sub:
        a['top1_same_lang'] += 1
    else:
        a['top1_diff_lang'] += 1
    # Save examples for weak languages
    if len(a['examples']) < 3 and r_top1['rouge1'].fmeasure < 0.4:
        a['examples'].append({
            'question': q[:200],
            'reference': ref[:200],
            'top1_answer': top1_answer[:200],
            'top1_subset': top1_sub,
            'oracle_answer': str(combined.iloc[best_ci]['output'])[:200],
            'oracle_subset': str(combined.iloc[best_ci]['subset']),
            'oracle_rank': best_rank,
            'top1_r1': r_top1['rouge1'].fmeasure,
            'oracle_r1': best_r1,
        })
# Print analysis
log(f"\n{'='*70}")
log(f"{'Subset':<10} {'N':>5} {'Top1 R1':>8} {'Orac R1':>8} {'Gap':>6} "
    f"{'OracRank':>9} {'SameLng%':>9} {'QLen':>5} {'ALen':>5} {'RefLen':>6}")
log(f"{'-'*70}")
for sub in sorted(analysis.keys()):
    a = analysis[sub]
    n = a['count']
    t1_r1 = np.mean(a['r1_top1'])
    or_r1 = np.mean(a['r1_oracle'])
    gap = or_r1 - t1_r1
    avg_rank = np.mean(a['oracle_rank'])
    same_pct = a['oracle_from_same_lang'] / max(n, 1) * 100
    qlen = np.mean(a['avg_q_len'])
    alen = np.mean(a['avg_a_len'])
    rlen = np.mean(a['avg_ref_len'])
    log(f"{sub:<10} {n:>5} {t1_r1:>8.3f} {or_r1:>8.3f} {gap:>+6.3f} "
        f"{avg_rank:>9.1f} {same_pct:>8.1f}% {qlen:>5.0f} {alen:>5.0f} {rlen:>6.0f}")
# Print examples for weak languages
for sub in ['Amh_Eth', 'Aka_Gha', 'Eng_Gha']:
    a = analysis[sub]
    if a['examples']:
        log(f"\n--- Examples for {sub} (why it's weak) ---")
        for j, ex in enumerate(a['examples'][:2]):
            log(f"\n  Example {j+1}:")
            log(f"    Q: {ex['question']}")
            log(f"    Reference:     {ex['reference']}")
            log(f"    Retrieved (R1={ex['top1_r1']:.3f}): {ex['top1_answer']}")
            log(f"      ↑ from subset: {ex['top1_subset']}")
            log(f"    Oracle (R1={ex['oracle_r1']:.3f}, rank={ex['oracle_rank']}): {ex['oracle_answer']}")
            log(f"      ↑ from subset: {ex['oracle_subset']}")
# ============================================================
# STEP 3: TEST PER-LANGUAGE RETRIEVAL STRATEGIES
# ============================================================
log(f"\n{'='*60}")
log("STEP 3: Per-language retrieval strategies")
log(f"{'='*60}")
def retrieve_answer(q_text, q_emb, subset, strategy):
    """Retrieve best answer using given strategy."""
    q_stripped = q_text.strip()
    if strategy == 'global':
        D, I = global_idx.search(q_emb.reshape(1, -1), 10)
        for j in range(10):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            if str(combined.iloc[ci]['input']).strip() == q_stripped: continue
            return str(combined.iloc[ci]['output'])
    elif strategy == 'same_lang':
        if subset not in lang_indices:
            return retrieve_answer(q_text, q_emb, subset, 'global')
        idx, mask = lang_indices[subset]
        D, I = idx.search(q_emb.reshape(1, -1), 10)
        for j in range(10):
            li = int(I[0][j])
            if li >= len(mask): continue
            ci = mask[li]
            if str(combined.iloc[ci]['input']).strip() == q_stripped: continue
            return str(combined.iloc[ci]['output'])
    elif strategy == 'same_lang_fallback':
        # Try same language first, fall back to global
        ans = retrieve_answer(q_text, q_emb, subset, 'same_lang')
        if ans:
            return ans
        return retrieve_answer(q_text, q_emb, subset, 'global')
    elif strategy == 'global_top3_longest':
        # Pick the LONGEST answer from top-3 (more content = potentially higher ROUGE recall)
        D, I = global_idx.search(q_emb.reshape(1, -1), 15)
        candidates = []
        for j in range(15):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            if str(combined.iloc[ci]['input']).strip() == q_stripped: continue
            candidates.append(str(combined.iloc[ci]['output']))
            if len(candidates) >= 3: break
        if candidates:
            return max(candidates, key=len)
    elif strategy == 'global_top3_best_overlap':
        # Pick answer from top-3 with most word overlap with the QUESTION
        D, I = global_idx.search(q_emb.reshape(1, -1), 15)
        q_words = set(q_stripped.lower().split())
        best_answer, best_overlap = '', 0
        count = 0
        for j in range(15):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            if str(combined.iloc[ci]['input']).strip() == q_stripped: continue
            ca = str(combined.iloc[ci]['output'])
            a_words = set(ca.lower().split())
            overlap = len(q_words & a_words)
            if overlap > best_overlap or not best_answer:
                best_overlap = overlap
                best_answer = ca
            count += 1
            if count >= 3: break
        return best_answer
    elif strategy == 'global_concat_top2':
        # Concatenate top-2 answers
        D, I = global_idx.search(q_emb.reshape(1, -1), 15)
        answers = []
        for j in range(15):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            if str(combined.iloc[ci]['input']).strip() == q_stripped: continue
            answers.append(str(combined.iloc[ci]['output']))
            if len(answers) >= 2: break
        return ' '.join(answers)
    return "No answer found."
strategies = [
    'global',               # current approach
    'same_lang',            # only same language
    'same_lang_fallback',   # same lang first, then global
    'global_top3_longest',  # longest of top-3
    'global_top3_best_overlap',  # most Q-word overlap from top-3
    'global_concat_top2',   # concatenate top-2
]
# Evaluate each strategy per language
strat_results = {}  # (strategy, subset) -> (r1, rl)
for strat in strategies:
    log(f"\nTesting: {strat}")
    per_lang = defaultdict(lambda: {'r1': [], 'rl': []})
    for i in tqdm(range(len(val_df)), desc=f"  {strat}"):
        q = str(val_df.iloc[i]['input']).strip()
        ref = str(val_df.iloc[i]['output']).strip()
        sub = str(val_df.iloc[i]['subset'])
        if not ref: continue
        answer = retrieve_answer(q, val_emb[i], sub, strat)
        if not answer:
            answer = "No answer found."
        r = scorer.score(ref, answer)
        per_lang[sub]['r1'].append(r['rouge1'].fmeasure)
        per_lang[sub]['rl'].append(r['rougeL'].fmeasure)
    for sub in per_lang:
        r1 = np.mean(per_lang[sub]['r1'])
        rl = np.mean(per_lang[sub]['rl'])
        strat_results[(strat, sub)] = (r1, rl)
# Find best strategy per language
log(f"\n{'='*60}")
log("BEST STRATEGY PER LANGUAGE")
log(f"{'='*60}")
best_per_lang = {}
log(f"\n{'Subset':<12} {'Best Strategy':<26} {'R1':>7} {'RL':>7} {'vs Global R1':>12}")
for sub in sorted(unique_subsets):
    best_strat, best_r1, best_rl = 'global', 0, 0
    for strat in strategies:
        if (strat, sub) in strat_results:
            r1, rl = strat_results[(strat, sub)]
            if r1 + rl > best_r1 + best_rl:
                best_strat, best_r1, best_rl = strat, r1, rl
    global_r1 = strat_results.get(('global', sub), (0, 0))[0]
    diff = best_r1 - global_r1
    best_per_lang[sub] = best_strat
    marker = " ★" if diff > 0.005 else ""
    log(f"  {sub:<12} {best_strat:<26} {best_r1:>7.4f} {best_rl:>7.4f} {diff:>+12.4f}{marker}")
# Calculate overall score with per-language strategy
log(f"\nOverall comparison:")
global_r1s, global_rls = [], []
perlang_r1s, perlang_rls = [], []
for i in range(len(val_df)):
    q = str(val_df.iloc[i]['input']).strip()
    ref = str(val_df.iloc[i]['output']).strip()
    sub = str(val_df.iloc[i]['subset'])
    if not ref: continue
    # Global
    ans_g = retrieve_answer(q, val_emb[i], sub, 'global')
    r_g = scorer.score(ref, ans_g or '')
    global_r1s.append(r_g['rouge1'].fmeasure)
    global_rls.append(r_g['rougeL'].fmeasure)
    # Per-language best
    best_strat = best_per_lang.get(sub, 'global')
    ans_p = retrieve_answer(q, val_emb[i], sub, best_strat)
    r_p = scorer.score(ref, ans_p or '')
    perlang_r1s.append(r_p['rouge1'].fmeasure)
    perlang_rls.append(r_p['rougeL'].fmeasure)
g_r1, g_rl = np.mean(global_r1s), np.mean(global_rls)
p_r1, p_rl = np.mean(perlang_r1s), np.mean(perlang_rls)
log(f"  {'Global (current)':<25} R1={g_r1:.4f} RL={g_rl:.4f}")
log(f"  {'Per-language best':<25} R1={p_r1:.4f} RL={p_rl:.4f}")
log(f"  {'Improvement':<25} R1={p_r1-g_r1:+.4f} RL={p_rl-g_rl:+.4f}")
# ============================================================
# STEP 4: GENERATE SUBMISSIONS
# ============================================================
log(f"\n{'='*60}")
log("STEP 4: Generate submissions")
log(f"{'='*60}")
# Re-load model for test encoding
bienc = SentenceTransformer(str(AFRIE5_DIR) if AFRIE5_DIR and AFRIE5_DIR.exists()
    else 'McGill-NLP/AfriE5-Large-instruct', device='cuda:0')
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
log("Encoding test queries...")
test_emb = bienc.encode(
    [f"{PREFIX}{q}" for q in test_qs],
    batch_size=64, show_progress_bar=True, normalize_embeddings=True
).astype(np.float32)
bienc.cpu(); gc.collect(); torch.cuda.empty_cache()
# Submission 1: Global (baseline)
log("\nGenerating baseline submission...")
rows_global = []
for i in tqdm(range(len(test_df)), desc="Global"):
    answer = retrieve_answer(test_qs[i], test_emb[i], test_subs[i], 'global')
    rows_global.append({
        'ID': test_df.iloc[i]['ID'],
        'TargetRLF1': answer, 'TargetR1F1': answer, 'TargetLLM': answer,
    })
sub_global = pd.DataFrame(rows_global)[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
sub_global.to_csv(OUTPUT_DIR / 'submission_global_baseline.csv', index=False)
log("Saved: submission_global_baseline.csv")
# Submission 2: Per-language best strategy
if p_r1 + p_rl > g_r1 + g_rl + 0.001:
    log("\nGenerating per-language submission...")
    rows_perlang = []
    for i in tqdm(range(len(test_df)), desc="Per-lang"):
        sub = test_subs[i]
        strat = best_per_lang.get(sub, 'global')
        answer = retrieve_answer(test_qs[i], test_emb[i], sub, strat)
        rows_perlang.append({
            'ID': test_df.iloc[i]['ID'],
            'TargetRLF1': answer, 'TargetR1F1': answer, 'TargetLLM': answer,
        })
    sub_pl = pd.DataFrame(rows_perlang)[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
    sub_pl.to_csv(OUTPUT_DIR / 'submission_perlang_best.csv', index=False)
    log("Saved: submission_perlang_best.csv")
else:
    log("Per-language strategy didn't improve. Skipping.")
# Submission 3: Per-column optimized
# ROUGE columns get retrieval, LLM column gets the longest/most detailed answer
log("\nGenerating per-column submission (ROUGE=top1, LLM=most detailed)...")
rows_percol = []
for i in tqdm(range(len(test_df)), desc="Per-col"):
    q = test_qs[i]
    sub = test_subs[i]
    # ROUGE columns: best strategy for this language
    strat = best_per_lang.get(sub, 'global')
    ans_rouge = retrieve_answer(q, test_emb[i], sub, strat)
    # LLM column: longest of top-3 (more comprehensive = better LLM judge)
    ans_llm = retrieve_answer(q, test_emb[i], sub, 'global_top3_longest')
    rows_percol.append({
        'ID': test_df.iloc[i]['ID'],
        'TargetR1F1': ans_rouge,
        'TargetRLF1': ans_rouge,
        'TargetLLM': ans_llm or ans_rouge,
    })
sub_pc = pd.DataFrame(rows_percol)[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
sub_pc.to_csv(OUTPUT_DIR / 'submission_percol_optimized.csv', index=False)
log("Saved: submission_percol_optimized.csv")
# ============================================================
# STEP 5: GEMINI API FOR LLM COLUMN (if available)
# ============================================================
log(f"\n{'='*60}")
log("STEP 5: Gemini API for LLM column (optional)")
log(f"{'='*60}")
gemini_available = False
try:
    api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
    if not api_key:
        # Try Colab secrets
        try:
            from google.colab import userdata
            api_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
        except Exception:
            pass
    if api_key:
        from google import genai
        client = genai.Client(api_key=api_key)
        gemini_available = True
        log("Gemini API available!")
    else:
        log("No Gemini API key found. Skipping Gemini generation.")
        log("To enable: set GEMINI_API_KEY in Colab secrets or environment.")
except Exception as e:
    log(f"Gemini not available: {e}")
if gemini_available:
    import time
    SUBSET_TO_LANG = {
        'Aka_Gha': 'Akan (Ghana)', 'Amh_Eth': 'Amharic (Ethiopia)',
        'Eng_Eth': 'English (Ethiopia)', 'Eng_Gha': 'English (Ghana)',
        'Eng_Ken': 'English (Kenya)', 'Eng_Uga': 'English (Uganda)',
        'Lug_Uga': 'Luganda (Uganda)', 'Swa_Ken': 'Swahili (Kenya)',
    }
    # Progress file for resuming
    progress_file = OUTPUT_DIR / 'gemini_llm_progress.json'
    if progress_file.exists():
        with open(progress_file) as f:
            gemini_answers = json.load(f)
        log(f"Resuming: {len(gemini_answers)} already done")
    else:
        gemini_answers = {}
    for i in tqdm(range(len(test_df)), desc="Gemini LLM"):
        row_id = str(test_df.iloc[i]['ID'])
        if row_id in gemini_answers:
            continue
        q = test_qs[i]
        sub = test_subs[i]
        lang = SUBSET_TO_LANG.get(sub, sub)
        # Get top-3 retrieved answers as context
        D, I = global_idx.search(test_emb[i:i+1], 10)
        contexts = []
        for j in range(10):
            ci = int(I[0][j])
            if ci >= len(combined): continue
            if str(combined.iloc[ci]['input']).strip() == q.strip(): continue
            contexts.append(str(combined.iloc[ci]['output']))
            if len(contexts) >= 3: break
        context_str = "\n".join([f"{k+1}. {c}" for k, c in enumerate(contexts)])
        prompt = f"""You are a multilingual health expert. Answer this health question accurately and comprehensively.
Language: {lang}
Question: {q}
Reference answers for context:
{context_str}
Instructions:
- Answer in {lang} (same language as the question)
- Be thorough, accurate, and culturally appropriate
- Include all relevant medical information
- Use clear, professional language
- Do NOT add disclaimers or meta-commentary"""
        for attempt in range(3):
            try:
                response = client.models.generate_content(
                    model='gemini-2.0-flash',
                    contents=prompt,
                    config=genai.types.GenerateContentConfig(
                        temperature=0.3,
                        max_output_tokens=512,
                    )
                )
                gemini_answers[row_id] = response.text.strip()
                break
            except Exception as e:
                if attempt < 2:
                    time.sleep(2 ** attempt)
                else:
                    gemini_answers[row_id] = contexts[0] if contexts else "No answer."
        # Rate limit (free tier: 15 RPM)
        time.sleep(4.5)
        # Save progress every 50
        if (i + 1) % 50 == 0:
            with open(progress_file, 'w') as f:
                json.dump(gemini_answers, f)
            log(f"  Progress saved: {len(gemini_answers)}/{len(test_df)}")
    # Save final progress
    with open(progress_file, 'w') as f:
        json.dump(gemini_answers, f)
    # Create Gemini hybrid submission
    rows_gemini = []
    for i in range(len(test_df)):
        row_id = str(test_df.iloc[i]['ID'])
        sub = test_subs[i]
        strat = best_per_lang.get(sub, 'global')
        ans_rouge = retrieve_answer(test_qs[i], test_emb[i], sub, strat)
        rows_gemini.append({
            'ID': test_df.iloc[i]['ID'],
            'TargetR1F1': ans_rouge,
            'TargetRLF1': ans_rouge,
            'TargetLLM': gemini_answers.get(row_id, ans_rouge),
        })
    sub_gemini = pd.DataFrame(rows_gemini)[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
    sub_gemini.to_csv(OUTPUT_DIR / 'submission_gemini_hybrid.csv', index=False)
    log("Saved: submission_gemini_hybrid.csv")
# ============================================================
# FINAL SUMMARY
# ============================================================
log(f"\n{'='*60}")
log("FINAL SUMMARY")
log(f"{'='*60}")
log(f"\nVal scores:")
log(f"  Global baseline:    R1={g_r1:.4f} RL={g_rl:.4f}")
log(f"  Per-language best:  R1={p_r1:.4f} RL={p_rl:.4f} ({p_r1-g_r1:+.4f}/{p_rl-g_rl:+.4f})")
log(f"\nSubmissions:")
for f in sorted(OUTPUT_DIR.glob("submission_*.csv")):
    log(f"  → {f.name}")
log(f"\nStrategy per language: {json.dumps(best_per_lang, indent=2)}")
log(f"\nPrevious best LB: 0.6545")
log(f"\nRecommended submit order:")
log(f"  1. submission_percol_optimized.csv (retrieval for ROUGE, longest for LLM)")
if gemini_available:
    log(f"  2. submission_gemini_hybrid.csv (retrieval for ROUGE, Gemini for LLM)")
log(f"  3. submission_perlang_best.csv (per-language retrieval)")
log("Done!")


[17:08:36] Google Drive already mounted.
[17:08:36] Loading data...
[17:08:37] Train: 29815, Val: 6686, Test: 2618, Combined: 36501
[17:08:37] 
[17:08:37] STEP 1: Load model + Build global and per-language indices
[17:08:37] ============================================================


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[17:08:41] AfriE5 from Drive: 560M params
[17:08:41] Encoding corpus...


Batches:   0%|          | 0/571 [00:00<?, ?it/s]

[17:10:04] Global index: (36501, 1024)
[17:10:04]   Aka_Gha: 5569 samples
[17:10:04]   Amh_Eth: 2307 samples
[17:10:04]   Eng_Eth: 4479 samples
[17:10:04]   Eng_Gha: 5547 samples
[17:10:04]   Eng_Ken: 2470 samples
[17:10:04]   Eng_Uga: 9312 samples
[17:10:04]   Lug_Uga: 4229 samples
[17:10:04]   Swa_Ken: 2588 samples
[17:10:04] Encoding val queries...


...

[17:34:12] Saved: submission_global_baseline.csv
[17:34:12] 
Generating per-language submission...


Per-lang: 100%|██████████| 2618/2618 [00:12<00:00, 205.27it/s]


[17:34:25] Saved: submission_perlang_best.csv
[17:34:25] 
Generating per-column submission (ROUGE=top1, LLM=most detailed)...


Per-col: 100%|██████████| 2618/2618 [00:33<00:00, 78.92it/s] 


[17:34:59] Saved: submission_percol_optimized.csv
[17:34:59] 
[17:34:59] STEP 5: Gemini API for LLM column (optional)
[17:34:59] ============================================================
[17:35:00] No Gemini API key found. Skipping Gemini generation.
[17:35:00] To enable: set GEMINI_API_KEY in Colab secrets or environment.
[17:35:00] 
[17:35:00] FINAL SUMMARY
[17:35:00] ============================================================
[17:35:00] 
Val scores:
[17:35:00]   Global baseline:    R1=0.6293 RL=0.5727
[17:35:00]   Per-language best:  R1=0.6383 RL=0.5778 (+0.0090/+0.0051)
[17:35:00] 
Submissions:
[17:35:00]   → submission_baseline.csv
[17:35:00]   → submission_baseline_qq.csv
[17:35:00]   → submission_ce_reranked.csv
[17:35:00]   → submission_fusion_best.csv
[17:35:00]   → submission_fusion_percol.csv
[17:35:00]   → submission_global_baseline.csv
[17:35:00]   → submission_hybrid.csv
[17:35:00]   → submission_percol_optimized.csv
[17:35:00]   → submission_perlang_best.csv
[17:35:0